# Exercise 2
## Maximizing the benefit
with restricted capacity in each team

In [1]:
#Exercise 2 - INNOVATIVE INITIATIVES
#Question 5 
# -> maximizing benefit
using GLPK, JuMP

J = 1:15                  # initiatives
T = 1:3                   # teams

b = [12, 24, 20, 28, 10, 11, 30, 27, 22, 25, 40, 38, 32, 34, 36]        # Strategic benefit
a = [14,  8,  7, 10, 13, 12,  9,  8,  7,  8, 12, 11, 10,  9, 10]               # Resource consumption
C = [28, 32, 35]    # Team capacities

# Dependency pairs:
dependencies = [    # (i,j) means initiative i requires initiative j
    (2,1),
    (3,1),
    (4,1),
    (7,5),
    (8,5),
    (9,6),
    (10,6),
    (11,7),
    (12,8),
    (15,10)]

model = Model(GLPK.Optimizer)

@variable(model,x[T,J],Bin)
@objective(model, Max, sum( x[i,j] * b[j] for i=T,j=J))

@constraint(model, [j=J], sum( x[i,j] for i=T) <= 1)
@constraint(model, [i=T], sum( x[i,j] * a[j] for j=J) <= C[i])

for dep in dependencies
    @constraint(model, sum( x[i,dep[1]] for i=T)  <= sum( x[i,dep[2]] for i=T) )
end

@constraint(model, sum( x[i,11] + x[i,12] for i=T) <= 1)
@constraint(model, sum( x[i,2] + x[i,3] + x[i,4] + x[i,7] + x[i,8] for i=T ) >= 3)

optimize!(model)

println("solver status: ", termination_status(model))
println("solution status: ", primal_status(model))
println("object value: ", objective_value(model))

print("\n")
for i=T
    for j=J
        print(value(x[i,j]))
        print(" ")
    end
    print("\n")
end

solver status: OPTIMAL
solution status: FEASIBLE_POINT
object value: 244.0

0.0 1.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 1.0 
1.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 0.0 
0.0 0.0 1.0 0.0 0.0 1.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 1.0 0.0 


## Lagrangian Relaxation

We added one constraint to the objective function, by introducing an additional constant $u$
$$
Z(u) = max \sum_{i=1}^{3}\sum_{j=1}^{15} x_{ij}b_j + \sum_{i=1}^{3}\lambda_i (C_i - \sum_{i=1}^3 x_{ij} a_j)
$$
and then minimized for $u$
$$
\min_{u} Z(u) 
$$


In [18]:
#Exercise 2
#Question 6/7 and first part of 8   
# Lagrange Dual solution with gradient-descent:  min_u(max_x Z(u)) 
# STEPSIZE: 1/k (last code with Polyak!)
using GLPK, JuMP
J = 1:15                  # initiatives
T = 1:3                   # teams

b = [12, 24, 20, 28, 10, 11, 30, 27, 22, 25, 40, 38, 32, 34, 36]        # Strategic benefit
a = [14,  8,  7, 10, 13, 12,  9,  8,  7,  8, 12, 11, 10,  9, 10]               # Resource consumption
C = [28, 32, 35]    # Team capacities
lambda = [2.9,3.8,1.03]
x_solution = zeros((length(T),length(J)))

# Dependency pairs:
dependencies = [    # (i,j) means initiative i requires initiative j
    (2,1),
    (3,1),
    (4,1),
    (7,5),
    (8,5),
    (9,6),
    (10,6),
    (11,7),
    (12,8),
    (15,10)]

lambda = zeros(3)
bestUpperBound = Inf
bestLambda = copy(lambda)
maxIter = 500 

for k=1:maxIter
    # OPTIMIZATION MODEL
    model = Model(GLPK.Optimizer)

    # @variable(model,1>=x[T,J]>=0)
    @variable(model,x[T,J],Bin)

    @objective(model, Max, 
    
        sum( x[i,j] * b[j] for i=T,j=J)
        +
        sum( lambda[i] * (C[i] - sum( x[i,j] * a[j] for j=J) ) for i=T))

    @constraint(model, [j=J], sum( x[i,j] for i=T) <= 1)
    # @constraint(model, [i=T], sum( x[i,j] * a[j] for j=J) <= C[i])

    for dep in dependencies
        @constraint(model, sum( x[i,dep[1]] for i=T)  <= sum( x[i,dep[2]] for i=T) )
    end

    @constraint(model, sum( x[i,11] + x[i,12] for i=T) <= 1)
    @constraint(model, sum( x[i,2] + x[i,3] + x[i,4] + x[i,7] + x[i,8] for i=T ) == 3)

    optimize!(model)

    # UPDATING UPPER BOUND and LAMBDA
    UB = objective_value(model)
    if UB < bestUpperBound
        bestUpperBound = UB
        bestLambda = copy(lambda)
        x_solution = value.(x)
    end

    # CALC NEW LAMBDA 
    for i=T
        mu = 1/(k)    #0.05/sqrt(k)
        lambda[i] = max(lambda[i]  -  mu * ( C[i] - sum( value(x[i,j]) * a[j] for j=J) ), 0 )
    end
    # println("solver status: ", termination_status(model))
    # println("solution status: ", primal_status(model))
    if k % 50 == 0        
        println(
        "k = ", k,
        "   UB = ", round(UB,digits=2),
        "   best UB = ", round(bestUpperBound,digits=5),
        "   lambda = ", round.(lambda,digits=2))
    end
end

println("BestLagrangianUB = ", bestUpperBound)
println("BestLambda = ", bestLambda)

println("x_solution = ")
display(x_solution)

k = 50   UB = 306.79   best UB = 275.38746   lambda = [2.78, 3.71, 2.83]
k = 100   UB = 284.28   best UB = 256.13721   lambda = [3.24, 2.51, 2.64]
k = 150   UB = 265.36   best UB = 256.13721   lambda = [2.55, 2.69, 2.87]
k = 200   UB = 259.76   best UB = 252.95566   lambda = [2.73, 2.76, 2.37]
k = 250   UB = 250.88   best UB = 248.82414   lambda = [2.6, 2.78, 2.44]
k = 300   UB = 256.72   best UB = 246.11823   lambda = [2.56, 2.56, 2.65]
k = 350   UB = 254.76   best UB = 246.11823   lambda = [2.53, 2.49, 2.59]
k = 400   UB = 255.33   best UB = 244.60414   lambda = [2.59, 2.54, 2.48]
k = 450   UB = 249.8   best UB = 244.60414   lambda = [2.43, 2.57, 2.61]
k = 500   UB = 250.31   best UB = 244.52918   lambda = [2.49, 2.6, 2.52]
BestLagrangianUB = 244.5291809956211
BestLambda = [2.535288086317492, 2.5396368919172274, 2.5287877660436653]
x_solution = 


2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, 1:3
    Dimension 2, 1:15
And data, a 3×15 Matrix{Float64}:
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  1.0  1.0  1.0  0.0  1.0  0.0  0.0  1.0  1.0  0.0  0.0  1.0  1.0  1.0

## LP-Relaxation
letting the variables take real-values, not just integers and thereby receiving an upper bound, which is generally infeasible

In [ ]:
#Exercise 2
#Question 8 (second part) - LP-Relaxation (positive real valued)
using GLPK, JuMP

J = 1:15                  # initiatives
T = 1:3                   # teams

b = [12, 24, 20, 28, 10, 11, 30, 27, 22, 25, 40, 38, 32, 34, 36]        # Strategic benefit
a = [14,  8,  7, 10, 13, 12,  9,  8,  7,  8, 12, 11, 10,  9, 10]        # Resource consumption
C = [28, 32, 35]    # Team capacities

# Dependency pairs:
dependencies = [    # (i,j) means initiative i requires initiative j
    (2,1),
    (3,1),
    (4,1),
    (7,5),
    (8,5),
    (9,6),
    (10,6),
    (11,7),
    (12,8),
    (15,10)]

model = Model(GLPK.Optimizer)

# ----- HERE WE SET THE VARIABLE TO REAL-VALUED INSTEAD OF INTEGER!!!! -----
@variable(model,x[T,J]>=0)
# -----

@objective(model, Max, sum( x[i,j] * b[j] for i=T,j=J))

@constraint(model, [j=J], sum( x[i,j] for i=T) <= 1)
@constraint(model, [i=T], sum( x[i,j] * a[j] for j=J) <= C[i])

for dep in dependencies
    @constraint(model, sum( x[i,dep[1]] for i=T)  <= sum( x[i,dep[2]] for i=T) )
end

@constraint(model, sum( x[i,11] + x[i,12] for i=T) <= 1)
@constraint(model, sum( x[i,2] + x[i,3] + x[i,4] + x[i,7] + x[i,8] for i=T ) >= 3)

optimize!(model)

println("solver status: ", termination_status(model))
println("solution status: ", primal_status(model))
println("object value: ", objective_value(model))

print("\n")
for i=T
    for j=J
        print(value(x[i,j]))
        print(" ")
    end
    print("\n")
end

solver status: OPTIMAL
solution status: FEASIBLE_POINT
object value: 254.89189189189193

0.33333333333333315 0.33333333333333315 0.33333333333333315 0.0 0.7817047817047819 0.0 0.0 1.0000000000000004 0.0 0.0 6.504336911945362e-17 0.015561015561015568 0.0 0.0 0.0 
0.0 0.0 0.0 0.33333333333333315 0.0 0.5945945945945946 0.0 0.0 0.0 0.5945945945945946 0.0 0.9844389844389849 0.0 0.0 0.5945945945945946 
0.0 0.0 0.0 0.0 0.21829521829521836 0.0 1.0000000000000002 0.0 0.5945945945945946 0.0 0.0 0.0 1.0 1.0 0.0 
